# Construction and Architectural Document Processing Pipeline

## Setup env

In [32]:
import os
import getpass
from uuid import uuid4

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE8 - Certification Challenge - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

# needed for running async code in notebook
import nest_asyncio
nest_asyncio.apply()

## Data ingestion

In [1]:
# Env setup
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional

# Configure logging for debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set up paths
PROJECT_ROOT = Path.cwd().parent  # Go up one level from notebooks/
DATA_DIR = PROJECT_ROOT / "data"
PARSED_DATA_DIR = DATA_DIR / "parsed"

### Token counting helpers

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4") -> int:
    """
    Count the number of tokens in text using tiktoken.
    
    Args:
        text: The text to count tokens for
        model: The OpenAI model to use for tokenization (default: gpt-4)
    
    Returns:
        Number of tokens
    """
    try:
        encoding = tiktoken.encoding_for_model(model)
        return len(encoding.encode(text))
    except KeyError:
        # Fallback to cl100k_base encoding (used by GPT-4, GPT-3.5-turbo, etc.)
        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))

def analyze_document_tokens_with_limits(full_text: str, model: str = "gpt-4-turbo") -> Dict[str, Any]:
    """
    Analyze token count for a document with model limit checking.
    """
    # Model limits (input + output combined)
    MODEL_LIMITS = {
        "gpt-4": 8192,
        "gpt-4-turbo": 128000,
        "gpt-4o": 128000,
        "gpt-4o-mini": 128000,
        "gpt-4.1": 1000000
    }
    
    total_tokens = count_tokens(full_text, model)
    
    model_limit = MODEL_LIMITS.get(model, 128000)  # Default to GPT-4 Turbo limit
    reserved_output_tokens = 2000  # Reserve tokens for response
    
    token_analysis = {
        "model_used": model,
        "total_tokens": total_tokens,
        "model_limit": model_limit,
        "effective_input_limit": model_limit - reserved_output_tokens,
        "fits_in_context": total_tokens <= (model_limit - reserved_output_tokens),
        "utilization_percentage": (total_tokens / (model_limit - reserved_output_tokens)) * 100,
        "recommendation": ""
    }
    
    # Generate recommendations
    if token_analysis["fits_in_context"]:
        if token_analysis["utilization_percentage"] < 50:
            token_analysis["recommendation"] = "✅ Document fits comfortably in context window"
        elif token_analysis["utilization_percentage"] < 80:
            token_analysis["recommendation"] = "⚠️ Document uses significant portion of context window"
        else:
            token_analysis["recommendation"] = "🔶 Document uses most of context window - consider chunking for complex tasks"
    else:
        token_analysis["recommendation"] = f"❌ Document exceeds context limit by {total_tokens - token_analysis['effective_input_limit']:,} tokens - chunking required"
    
    return token_analysis

def print_token_analysis_with_limits(token_analysis: Dict[str, Any]):
    """Print formatted token analysis with limit information."""
    if "error" in token_analysis:
        print(f"❌ {token_analysis['error']}")
        return
    
    print("📊 Token Analysis with Model Limits")
    print("=" * 60)
    print(f"Model: {token_analysis['model_used']}")
    print(f"Document tokens: {token_analysis['total_tokens']:,}")
    print(f"Model limit: {token_analysis['model_limit']:,}")
    print(f"Effective input limit: {token_analysis['effective_input_limit']:,}")
    print(f"Utilization: {token_analysis['utilization_percentage']:.1f}%")
    print(f"Fits in context: {'Yes' if token_analysis['fits_in_context'] else 'No'}")
    print(f"Recommendation: {token_analysis['recommendation']}")

### Document Sectionizer and Chunker logic

In [26]:
import re
import json
import uuid
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Callable, Tuple

# ---------------------------------------
# Sectionizer data model
# ---------------------------------------
@dataclass
class Section:
    id: str
    level: int                      # 1 = PART, 2 = ARTICLE, 3 = sub-article (optional)
    header: str
    header_path: List[str]
    page_start: Optional[int]       # Unknown from MD; keep None unless you map later
    page_end: Optional[int]
    text: str                       # concatenated body text for this section

@dataclass
class SectionChunk:
    section_id: str
    header_path: List[str]
    chunk_index: int
    text: str                       # chunked text within the section

# ---------------------------------------
# CSI-aware heuristics
# ---------------------------------------
PART_RE = re.compile(r'^\s*PART\s+(?:[123]|I|II|III)\s*-\s+.+$', re.IGNORECASE)

# ALL-CAPS-ish line (len-limited)
ALLCAPS_RE = re.compile(r'^[A-Z0-9][A-Z0-9 \-/,()&\.]{3,}$')

# Numbered article headings like "1.1 SUMMARY", "2.3 MATERIALS"
NUM_ARTICLE_RE = re.compile(r'^\s*(?P<part>[1-3])\.(?P<art>\d+)\s+(?P<title>[A-Z][A-Z0-9 \-/,()&\.]{2,})\s*$')

# Things that look like list items, not headings: "1.", "1.1.", "a)", "-"
LIST_LIKE_RE = re.compile(r'^\s*(?:\d+(?:\.\d+)*[\.\)]|[a-zA-Z][\.\)])\s+')

CSI_ARTICLE_HINTS = {
    "SUMMARY","REFERENCES","SUBMITTALS","QUALITY ASSURANCE",
    "DELIVERY, STORAGE, AND HANDLING","SEQUENCING","WARRANTY",
    "PERFORMANCE REQUIREMENTS","SYSTEM DESCRIPTION","ELEVATORS",
    "MATERIALS","MANUFACTURERS","PRODUCTS","EXECUTION","INSTALLATION",
    "FIELD QUALITY CONTROL","CAR ENCLOSURES","HOISTWAY ENTRANCES",
    "OPERATION","CAR FIXTURES","HALL FIXTURES","DEFINITIONS"
}

def _looks_like_markdown_heading(line: str):
    m = re.match(r'^(#{1,6})\s+(.+?)\s*$', line.strip())
    if m:
        return True, len(m.group(1)), m.group(2).strip()
    return False, 0, ""

def _normalize_header(text: str) -> str:
    return re.sub(r'\s+', ' ', text.strip())

def _slugify(parts: List[str]) -> str:
    import re as _re
    s = "-".join(parts).lower()
    return _re.sub(r'[^a-z0-9]+', '-', s).strip('-')

def _normalize_header(text: str) -> str:
    return re.sub(r'\s+', ' ', text.strip())

def _slugify(parts: List[str]) -> str:
    s = "-".join(parts)
    s = s.lower()
    s = re.sub(r'[^a-z0-9]+', '-', s).strip('-')
    return s


def sectionize_markdown(md_text: str) -> List[Section]:
    """
    Build sections from Docling-exported Markdown (CSI-aware):
    - PART detection has highest priority (even if it's a Markdown heading).
    - Numbered articles 'x.y TITLE' are anchored under PART x (imputed if missing).
    - ALLCAPS/known-article headings become Article level.
    """
    lines = md_text.splitlines()
    sections: List[Section] = []
    path_stack: List[Tuple[int, str]] = []  # (level, header)
    current: Optional[Section] = None

    def start_section(level: int, header: str):
        nonlocal current, path_stack, sections
        header = _normalize_header(header)
        # pop to parent
        while path_stack and path_stack[-1][0] >= level:
            path_stack.pop()
        path_stack.append((level, header))
        header_path = [h for _, h in path_stack]
        sec_id = f"sec-{_slugify(header_path)}-{uuid.uuid4().hex[:8]}"
        current = Section(
            id=sec_id, level=level, header=header,
            header_path=header_path, page_start=None, page_end=None, text=""
        )
        sections.append(current)

    def ensure_part(part_no: str):
        """Ensure top of stack is PART <part_no>; create an imputed PART if needed."""
        # If current top-level PART is already correct, nothing to do
        for lvl, hdr in reversed(path_stack):
            if lvl == 1:
                # try to detect the number in existing header
                m = re.search(r'\bPART\s+([1-3]|I|II|III)\b', hdr, re.IGNORECASE)
                if m:
                    cur = m.group(1)
                    # Normalize roman <-> arabic (simple)
                    if cur in {"I","II","III"}:
                        cur = {"I":"1","II":"2","III":"3"}[cur]
                    if cur == part_no:
                        return
                # wrong part at level 1 → pop it
                while path_stack and path_stack[-1][0] >= 1:
                    path_stack.pop()
                break
        # Create an imputed PART header if missing
        start_section(1, f"PART {part_no} - GENERAL (IMPUTED)")

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            # still attach whitespace to current text to preserve spacing a bit
            if current:
                current.text += "\n"
            continue

        # Check if this is a Markdown heading
        is_md, md_level, md_header = _looks_like_markdown_heading(line)

        # 1) PART has highest priority (even if it's a MD heading)
        if (is_md and PART_RE.match(md_header)) or PART_RE.match(line):
            start_section(1, md_header if is_md else line)
            continue

        # 2) Numbered article like "1.1 SUMMARY" → anchor to PART 1
        m_num = NUM_ARTICLE_RE.match(line.upper())
        if m_num and not LIST_LIKE_RE.match(line):
            ensure_part(m_num.group('part'))  # creates PART if missing
            start_section(2, f"{m_num.group('part')}.{m_num.group('art')} {m_num.group('title')}")
            continue

        # 3) Other Markdown headings (non-PART) → treat as Article/Sub-article
        if is_md:
            # If we have a PART already, make this level 2; else level 1
            level = 2 if any(l == 1 for l, _ in path_stack) else 1
            start_section(level, md_header)
            continue

        # 4) ALLCAPS/known-article headings → Article
        t = line.strip()
        if (t.upper() in CSI_ARTICLE_HINTS or
            (ALLCAPS_RE.match(t) and not LIST_LIKE_RE.match(t) and not t.endswith('.'))):
            level = 2 if any(l == 1 for l, _ in path_stack) else 1
            start_section(level, t)
            continue

        # 5) Content
        if current is None:
            start_section(1, "PREFACE")
        current.text += (line + "\n")

    # Trim text
    for s in sections:
        s.text = s.text.strip()
    return sections


# ---------------------------------------
# Chunker inside sections
# ---------------------------------------
def default_token_count(s: str) -> int:
    # ~4 chars/token rough heuristic
    return max(1, int(len(s) / 4))

def chunk_sections(
    sections: List[Section],
    max_tokens: int = 700,
    overlap_tokens: int = 80,
    token_counter: Optional[Callable[[str], int]] = None
) -> List[SectionChunk]:
    """
    Chunk each section's text to fit LLM limits, with overlap. 
    Splits on paragraphs/sentences when possible.
    """
    tc = token_counter or default_token_count
    chunks: List[SectionChunk] = []

    SENT_SPLIT = re.compile(r'(?<=[\.\:\;])\s+\n?|\n{2,}')  # sentence/paragraph-ish

    for s in sections:
        text = s.text or ""
        if not text.strip():
            continue

        parts = [p.strip() for p in SENT_SPLIT.split(text) if p.strip()]
        buf: List[str] = []
        buf_tokens = 0
        idx = 0

        def flush():
            nonlocal buf, buf_tokens, idx
            if not buf:
                return
            chunk_text = " ".join(buf).strip()
            chunks.append(SectionChunk(
                section_id=s.id,
                header_path=s.header_path,
                chunk_index=idx,
                text=chunk_text
            ))
            idx += 1
            # build overlap
            enc_len = tc(chunk_text)
            # crude: keep last N tokens by truncating characters proportionally
            if enc_len > overlap_tokens:
                keep_ratio = overlap_tokens / enc_len
                keep_chars = max(1, int(len(chunk_text) * keep_ratio))
                overlap_text = chunk_text[-keep_chars:]
            else:
                overlap_text = chunk_text
            buf = [overlap_text]
            buf_tokens = tc(overlap_text)

        for part in parts:
            p_tokens = tc(part)
            if buf_tokens + p_tokens > max_tokens and buf:
                flush()
            buf.append(part)
            buf_tokens += p_tokens

        if buf:
            # final flush without adding overlap
            chunk_text = " ".join(buf).strip()
            chunks.append(SectionChunk(
                section_id=s.id,
                header_path=s.header_path,
                chunk_index=idx,
                text=chunk_text
            ))

    return chunks

### Document conversion (with Docling)

In [27]:
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption
from docling.datamodel.pipeline_options import EasyOcrOptions
from docling_core.types.doc import ImageRefMode, PictureItem, TableItem
import pandas as pd

# Configure Docling pipeline options for construction documents
def create_docling_config(full_page_ocr: bool = False):
    """Create Docling configuration optimized for construction documents"""
    
    # PDF pipeline options based on real construction document analysis
    pdf_options = PdfPipelineOptions(
        do_table_structure=True,
        do_ocr=True,
        ocr_options=EasyOcrOptions(
            lang=["en"],  # Specify your language(s)
            confidence_threshold=0.7,  # Higher threshold for better quality
            use_gpu=True,  # Enable GPU acceleration if available
            recog_network="standard",  # Use standard recognition network
            force_full_page_ocr=full_page_ocr # processes each page purely via OCR (often slower than hybrid detection)
        ),
        # RapidOcrOptions(
        #     backend="onnxruntime",  # Fast inference backend
        #     text_score=0.6,  # Higher confidence threshold
        #     force_full_page_ocr=True
        # ),
        # TesseractOcrOptions(
        #     lang=["eng"],
        #     psm=6  # Uniform block of text (good for most documents)
        # )
        # generate_page_images=True,
        # generate_table_images=True,
        generate_picture_images=True,
        images_scale=2.0,  # Higher scale for better text recognition
        force_backend_text=False  # Let OCR generate the text
    )
    
    # Format options
    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
    
    return format_options


def parse_document_with_docling(
    converter: DocumentConverter, pdf_path: Path, is_csi_spec: bool = False,
    save_pictures: bool = False, save_tables: bool = False,
    write_artifacts: bool = True,
    token_counter: Optional[Callable[[str], int]] = None) -> Dict[str, Any]:
    """Parse a single PDF document using Docling"""

    document_name = pdf_path.stem
    print(f"🔄 Parsing {document_name}...")
    start_time = datetime.now()
    
    try:
        # Convert document
        conv_res = converter.convert(str(pdf_path))
        doc = conv_res.document
        
        # Extract basic document information
        md_text = doc.export_to_markdown()
        section_dicts = []
        section_chunk_dicts = []
        # sectionize and chunk CSI Spec
        if is_csi_spec:
            sections = sectionize_markdown(md_text)
            section_dicts = [asdict(s) for s in sections]
            section_chunks = chunk_sections(sections, max_tokens=700, overlap_tokens=80, token_counter=token_counter)
            section_chunk_dicts = [asdict(c) for c in section_chunks]

        doc_info: Dict[str, Any] = {
            "filename": pdf_path.name,
            "document_name": document_name,
            "parse_timestamp": start_time.isoformat(),
            "processing_time_seconds": (datetime.now() - start_time).total_seconds(),
            "success": True,
            "error": None,
            "full_text": md_text,
            "sections": section_dicts,
            "section_chunks": section_chunk_dicts,
            "tables": [],
            "figures": []
        }
        
        # Save the full text content to a Markdown file
        parsed_dir = Path(PARSED_DATA_DIR)
        parsed_dir.mkdir(parents=True, exist_ok=True)

        # Save artifacts (optional)
        if write_artifacts:
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            md_filename = parsed_dir / f"{document_name}_{ts}.md"
            doc.save_as_markdown(md_filename, image_mode=ImageRefMode.REFERENCED)

            if is_csi_spec:
                # Sections JSONL
                with (parsed_dir / f"{document_name}_{ts}.sections.jsonl").open("w", encoding="utf-8") as fp:
                    for s in section_dicts:
                        fp.write(json.dumps(s, ensure_ascii=False) + "\n")

                # Section Chunks JSONL
                with (parsed_dir / f"{document_name}_{ts}.section_chunks.jsonl").open("w", encoding="utf-8") as fp:
                    for c in section_chunk_dicts:
                        fp.write(json.dumps(c, ensure_ascii=False) + "\n")

        # Extract tables and figures (optional)
        if save_tables or save_pictures:
            table_counter = 0
            picture_counter = 0
            for element, _level in conv_res.document.iterate_items():
                if save_tables and isinstance(element, TableItem):
                    table_counter += 1
                    table_df: pd.DataFrame = element.export_to_dataframe()
                    logger.debug(f"## Table {table_counter}")
                    logger.debug(table_df.to_markdown())
                    table_info = {
                        "table_id": table_counter,
                        "markdown": table_df.to_markdown()
                    }
                    doc_info["tables"].append(table_info)
                    # Save the table as CSV
                    # element_csv_filename = parsed_dir / f"{document_name}-table-{table_ix + 1}.csv"
                    # logger.info(f"Saving CSV table to {element_csv_filename}")
                    # table_df.to_csv(element_csv_filename)

                if save_pictures and isinstance(element, PictureItem):
                    picture_counter += 1
                    figure_info = {
                        "figure_id": picture_counter,
                        "bbox": element.bbox if hasattr(element, 'bbox') else None,
                        "caption": element.caption if hasattr(element, 'caption') else None
                    }
                    doc_info["figures"].append(figure_info)
                    element_image_filename = parsed_dir / f"{document_name}-figure-{picture_counter}.png"
                    with element_image_filename.open("wb") as fp:
                        element.get_image(conv_res.document).save(fp, "PNG")
                
        print(f"✅ Successfully parsed {document_name}")
        print(f"   - Processing time: {doc_info['processing_time_seconds']:.2f} seconds")
        print(f"   - Sections: {len(doc_info['sections'])}")
        print(f"   - Section chunks: {len(doc_info['section_chunks'])}")
        print(f"   - Tables: {len(doc_info['tables'])}")
        print(f"   - Figures: {len(doc_info['figures'])}")
        
        return doc_info
        
    except Exception as e:
        error_info = {
            "filename": pdf_path.name,
            "document_name": document_name,
            "parse_timestamp": start_time.isoformat(),
            "processing_time_seconds": (datetime.now() - start_time).total_seconds(),
            "success": False,
            "error": str(e),
            "full_text": None,
            "sections": [],
            "section_chunks": [],
            "tables": [],
            "figures": [],
        }
        
        print(f"❌ Error parsing {document_name}: {e}")
        return error_info    

#### Pass A — Sectionizer
 * Goal: split text into logical chunks aligned to headings/bullets.

In [ ]:
print("🚀 Starting document parsing...")
print("=" * 80)

converter = DocumentConverter(format_options=create_docling_config())
spec_file = Path(DATA_DIR / "Spec 14 24 00 - Hydraulic Elevators_redacted.pdf")
# Use hybrid detection for this document
spec_doc = parse_document_with_docling(
    converter, 
    spec_file, 
    is_csi_spec=True,
    save_pictures=False, 
    save_tables=False, 
    write_artifacts=True, 
    token_counter=count_tokens)
print()

# Use full page OCR to get better results for the Submittal
# submittal_doc = parse_document_with_docling(
#     ocr_converter, 
#     submittal_file, 
#     save_pictures=False, 
#     save_tables=False, 
#     write_artifacts=False, 
#     token_counter=count_tokens)
# print()


2025-10-17 20:24:36,549 - docling.datamodel.document - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-17 20:24:36,551 - docling.document_converter - INFO - Going to convert document batch...
2025-10-17 20:24:36,551 - docling.document_converter - INFO - Initializing pipeline for StandardPdfPipeline with options hash 41e7b9a26929a2bda9240131dd84cd34


🚀 Starting document parsing...
🔄 Parsing Spec 14 24 00 - Hydraulic Elevators_redacted...


/Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/.venv/lib/python3.13/site-packages/docling/models/easyocr_model.py:68: UserWarning: Deprecated field. Better to set the `accelerator_options.device` in `pipeline_options`. When `use_gpu and accelerator_options.device == AcceleratorDevice.CUDA` the GPU is used to run EasyOCR. Otherwise, EasyOCR runs in CPU.
  warnings.warn(
2025-10-17 20:24:39,654 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-17 20:24:40,850 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-17 20:24:41,415 - docling.pipeline.base_pipeline - INFO - Processing document Spec 14 24 00 - Hydraulic Elevators_redacted.pdf
2025-10-17 20:24:47,072 - docling.document_converter - INFO - Finished converting document Spec 14 24 00 - Hydraulic Elevators_redacted.pdf in 10.53 sec.
2025-10-17 20:24:47,128 - docling.datamodel.document - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-17 20:24:47,142 - 

✅ Successfully parsed Spec 14 24 00 - Hydraulic Elevators_redacted
   - Processing time: 10.56 seconds
   - Sections: 38
   - Section chunks: 32
   - Tables: 0
   - Figures: 0

🔄 Parsing Submittal and Product Description_redacted...


2025-10-17 20:24:49,539 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-17 20:24:50,160 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-17 20:24:50,412 - docling.pipeline.base_pipeline - INFO - Processing document Submittal and Product Description_redacted.pdf
2025-10-17 20:28:45,987 - docling.document_converter - INFO - Finished converting document Submittal and Product Description_redacted.pdf in 238.86 sec.


✅ Successfully parsed Submittal and Product Description_redacted
   - Processing time: 238.90 seconds
   - Sections: 0
   - Section chunks: 0
   - Tables: 0
   - Figures: 0



#### Pass B — Open-schema Fact Harvest
 * defines a flexible **EAV** JSONL schema (**Entity-Attribute-Value** with evidence),

##### Json Fact Schema used as structure output for the LLM

```json
{
  "id": "uuid",
  "entity": { "type": "elevator", "name": "Endura MRL", "manufacturer": "ThyssenKrupp" },
  "attribute": { "raw": "Rated Speed (Up)", "canonical": null },
  "value": {
    "raw": "120 fpm",
    "type": "quantity",              // quantity | text | enum | boolean | range
    "num": 120,                      // for quantity
    "unit": "fpm",                   // for quantity
    "min": null, "max": null         // for range (if used)
  },
  "op": "=",                         // = | >= | <= | ~ | between
  "qualifiers": { "direction": "up" },
  "context": {
    "doc_id": "Spec_14_24_00.pdf",
    "section_id": "sec-part-2-products-elevators-…",
    "header_path": ["PART 2 - PRODUCTS","ELEVATORS","Elevator Description"],
    "source_span": "Rated Speed: a. Up: 120 fpm.",
    "confidence": 0.95
  }
}
```

##### Extraction Prompts

In [ ]:
FACT_EXTRACTOR_SYSTEM_PROMPT = """
You are a construction/architecture spec assistant. You understand CSI specs.
Act like a construction specialist and carefully extract explicit technical facts paying attention to technincal standards, codes and specifications. 
These facts will be further used to perform a comparison with submttal and product description (manufecture brochures, etc).
Do extract them as JSON Lines (one JSON object per line)!

Rules:
- NO inference: only facts explicitly stated in the text.
- Copy values verbatim into value.raw; if numeric, also parse value.num and value.unit.
- Use op in {"=",">=","<=","~","between"}; use "between" when a closed range is printed.
- Keep source_span ≤ 25 words, verbatim from the text.
- If multiple subfacts (e.g., Up/Down speeds), emit multiple JSON lines with qualifiers.
- If nothing factual is present, output nothing (empty response).
- Preserve factual meaning.
- Output ONLY JSON Lines. Do not wrap in arrays. No prose.
"""

EXTRACTOR_PROMPT_TEMPLATE = """DOC_ID: {doc_id}
SECTION_ID: {section_id}
HEADER_PATH: {header_path}

TEXT:
<<<
{chunk_text}
>>>

Emit JSON Lines with fields:
id (uuid), entity{{type?,name?,manufacturer?}}, attribute{{raw,canonical?}}, value{{raw,type,num?,unit?,min?,max?}}, op, qualifiers?, context{{doc_id,section_id,header_path,source_span,confidence}}.
"""


##### Fact extraction (LLM) + validation + (optional) normalization

The code below assumes you already have `doc_info["section_chunks"]` from the Sectionizer triggered by the `parse_document_with_docling`. 
 * `extract_facts_from_chunk`: calls the LLM and parses NDJSON safely.
 * `harvest_facts_for_doc`: iterates all chunks, adds doc/section context, de-dupes.
 * `normalize_units (optional)`: uses `pint` library to add normalized SI or preferred units.

In [ ]:
import json, uuid, re
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# ---------- Pydantic models for safety (optional but recommended)
from pydantic import BaseModel, Field, ValidationError, model_validator

class Entity(BaseModel):
    type: Optional[str] = None
    name: Optional[str] = None
    manufacturer: Optional[str] = None

class Attribute(BaseModel):
    raw: str
    canonical: Optional[str] = None

class Value(BaseModel):
    raw: str
    type: str                       # quantity | text | enum | boolean | range
    num: Optional[float] = None
    unit: Optional[str] = None
    min: Optional[float] = None
    max: Optional[float] = None

    @model_validator(mode="before")
    def check_value_fields(cls, v):
        t = v.get("type")
        if t == "quantity" and v.get("num") is None:
            # allow LLM to miss num; we'll backfill later if possible
            pass
        if t == "range" and (v.get("min") is None or v.get("max") is None):
            pass
        return v

class Context(BaseModel):
    doc_id: str
    section_id: str
    header_path: List[str]
    source_span: str
    confidence: float = Field(ge=0.0, le=1.0)

class Fact(BaseModel):
    id: str
    entity: Entity
    attribute: Attribute
    value: Value
    op: str = "="
    qualifiers: Optional[Dict[str, Any]] = None
    context: Context

# ---------- JSONL parsing helpers

def parse_jsonl(text: str) -> List[Dict[str, Any]]:
    """
    Parse NDJSON. Ignores blank lines and tolerates trailing code fences.
    """
    cleaned = text.strip()
    # Strip code fences if the model added them
    cleaned = re.sub(r"^```(?:jsonl|json)?\s*|```$", "", cleaned, flags=re.MULTILINE).strip()
    items = []
    for line in cleaned.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            items.append(json.loads(line))
        except json.JSONDecodeError:
            # Best-effort repair: try to fix trailing commas or single quotes
            line2 = line.replace("'", '"')
            try:
                items.append(json.loads(line2))
            except Exception:
                # Log and skip
                # print("Bad JSONL line:", line)
                continue
    return items

# ---------- LLM call (TODO: move to Ollama gpt-oss)
def llm_extract_jsonl(model: str, system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
    """
    Replace with your LLM client.
    Should return the raw text body (NDJSON lines).
    """
    try:
        client = ChatOpenAI(model=model, temperature=temperature)
        
        # Create messages list
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        response = client.invoke(messages)
        logger.info(f"\nLLM response: {response.content}\n")
        return response.content
        
    except Exception as e:
        logger.error(f"Error calling LLM: {e}")
        raise

# ---------- Extraction per chunk

def extract_facts_from_chunk(
    model: str,
    doc_id: str,
    chunk: Dict[str, Any],
    entity_hint: Optional[str] = None,
    temperature: float = 0.0
) -> List[Fact]:
    section_id = chunk["section_id"]
    header_path = chunk["header_path"]
    text = chunk["text"]

    system_prompt = FACT_EXTRACTOR_SYSTEM_PROMPT
    if entity_hint:
        system_prompt += f"\nEntity hint: The entity.type in this section is likely '{entity_hint}'."

    user_prompt = EXTRACTOR_PROMPT_TEMPLATE.format(
        doc_id=doc_id,
        section_id=section_id,
        header_path=" > ".join(header_path),
        chunk_text=text
    )

    raw = llm_extract_jsonl(model, system_prompt, user_prompt, temperature=temperature)
    items = parse_jsonl(raw)

    facts: List[Fact] = []
    for it in items:
        # fill required context bits if missing
        it.setdefault("id", str(uuid.uuid4()))
        it.setdefault("context", {})
        it["context"].setdefault("doc_id", doc_id)
        it["context"].setdefault("section_id", section_id)
        # Fix: Convert string header_path back to list if needed
        llm_header_path = it["context"].get("header_path", header_path)
        if isinstance(llm_header_path, str):
            # Split the string back into a list
            it["context"]["header_path"] = llm_header_path.split(" > ")
        else:
            # Use the original header_path from chunk if LLM didn't provide one
            it["context"]["header_path"] = header_path
            
        if "confidence" not in it["context"]:
            it["context"]["confidence"] = 0.9

        # basic resilience: ensure sub-objects exist
        it.setdefault("entity", {})
        it.setdefault("attribute", {"raw": ""})
        it.setdefault("value", {"raw": "", "type": "text"})

        try:
            facts.append(Fact(**it))
        except ValidationError as ve:
            # inspect ve.errors() to improve prompts
            logger.error(f"Validation error: {ve}")
            continue

    # Verify each fact against span before returning
    facts = self_verify_against_span(facts, chunk["text"])
    return facts

# ---------- Optional: basic unit normalization with pint

def normalize_units(facts: List[Fact]) -> List[Fact]:
    """
    Adds normalized units when value.type == 'quantity'.
    Example: fpm -> m/s; inches -> mm. Customize as needed.
    """
    try:
        import pint
        ureg = pint.UnitRegistry(autoconvert_offset_to_baseunit=True)
        # custom unit aliases common in specs
        ureg.define("fpm = foot / minute")
        ureg.define("inches = inch")
    except Exception:
        ureg = None

    if not ureg:
        return facts

    preferred = {
        "fpm": "fpm",      # keep native for elevator speeds
        "inch": "in",
        "inches": "in",
        "mm": "mm",
        "lb": "lb",
        "kg": "kg",
        # add more as needed
    }

    out: List[Fact] = []
    for f in facts:
        if f.value.type == "quantity" and f.value.num is not None and f.value.unit:
            unit = f.value.unit.strip().lower()
            # normalize some typographical variants
            unit = {"inches":"in", "inch":"in", "fpm":"fpm"}.get(unit, unit)
            try:
                q = (f.value.num) * ureg(unit)
                # choose target unit
                target = preferred.get(unit, unit)
                q2 = q.to(target)
                f.value.num = float(q2.magnitude)
                f.value.unit = target
            except Exception:
                pass
        out.append(f)
    return out

# ---------- Deduping

def fact_signature(f: Fact) -> str:
    """
    Used for dedupe: same entity + attribute + value.raw within the same section.
    """
    ent = (f.entity.type or "", f.entity.name or "", f.entity.manufacturer or "")
    attr = f.attribute.canonical or f.attribute.raw
    return "|".join([*ent, attr, f.value.raw, f.context.section_id])

def dedupe_facts(facts: List[Fact]) -> List[Fact]:
    seen = set()
    out = []
    for f in facts:
        sig = fact_signature(f)
        if sig in seen:
            continue
        seen.add(sig)
        out.append(f)
    return out

# ---------- Orchestrate all chunks for one doc

def harvest_facts_for_doc(
    model: str,
    doc_info: Dict[str, Any],
    entity_hints: Optional[Dict[str, str]] = None,  # {section_id: "elevator"}
    normalize: bool = False
) -> List[Dict[str, Any]]:
    """
    Iterates all section_chunks, extracts facts, (optionally) normalizes and dedupes.
    Returns list of JSON-serializable dicts (ready to write as JSONL).
    """
    doc_id = doc_info["filename"]
    chunks = doc_info.get("section_chunks", [])
    all_facts: List[Fact] = []

    for ch in chunks:
        hint = None
        if entity_hints:
            hint = entity_hints.get(ch["section_id"]) or entity_hints.get("default")
        facts = extract_facts_from_chunk(model, doc_id, ch, entity_hint=hint)
        all_facts.extend(facts)

    if normalize:
        all_facts = normalize_units(all_facts)

    all_facts = dedupe_facts(all_facts)

    # jsonify
    return [json.loads(f.model_dump_json()) for f in all_facts]

#  quick self-verification (cheap guardrail)
def self_verify_against_span(facts: List[Fact], chunk_text: str) -> List[Dict[str, Any]]:
    ok = []
    for f in facts:
        span = f.context.source_span 
        vr = f.value.raw 
        span_ok = span and (span in chunk_text)
        value_ok = (vr.lower() in span.lower()) if vr else True
        if span_ok and value_ok:
            ok.append(f)
        else:
            # keep but lower confidence or tag
            f.context.confidence = min(0.6, f.context.confidence)
            ok.append(f)
    return ok

##### Trigger CSI Spec Fact extraction via LLM

In [138]:
# Harvest facts
spec_facts = harvest_facts_for_doc(
    model="gpt-4o-mini" ,  # or your preferred deterministic model
    doc_info=spec_doc,
    entity_hints={"default": "elevator"},  # optional
    normalize=False #we'll normalize after parsing constraints
)

# Write JSONL
out_path = PARSED_DATA_DIR / (Path(spec_doc["document_name"]).stem + ".facts.jsonl")
with out_path.open("w", encoding="utf-8") as fp:
    for f in spec_facts:
        fp.write(json.dumps(f, ensure_ascii=False) + "\n")

2025-10-19 15:24:27,432 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 15:24:27,437 - __main__ - INFO - 
LLM response: {"id":"1","entity":{"type":"elevator","name":"Bonham ISD Middle School Additions and Renovations","manufacturer":null},"attribute":{"raw":"Location","canonical":"Location"},"value":{"raw":"Bonham, Texas","type":"string"},"op":"=","context":{"doc_id":"Spec 14 24 00 - Hydraulic Elevators_redacted.pdf","section_id":"sec-preface-b4fc6291","header_path":"PREFACE","source_span":"Bonham ISD Middle School Additions and Renovations Bonham, Texas","confidence":1}}

2025-10-19 15:24:30,018 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 15:24:30,020 - __main__ - INFO - 
LLM response: {"id":"1","entity":{"type":"elevator","name":"hydraulic passenger elevators"},"attribute":{"raw":"hydraulic passenger elevators"},"value":{"raw":"hydraulic passenger elevators","ty

In [151]:
len(spec_facts)
spec_facts

[{'id': '1',
  'entity': {'type': 'elevator',
   'name': 'Bonham ISD Middle School Additions and Renovations',
   'manufacturer': None},
  'attribute': {'raw': 'Location', 'canonical': 'Location'},
  'value': {'raw': 'Bonham, Texas',
   'type': 'string',
   'num': None,
   'unit': None,
   'min': None,
   'max': None},
  'op': '=',
  'qualifiers': None,
  'context': {'doc_id': 'Spec 14 24 00 - Hydraulic Elevators_redacted.pdf',
   'section_id': 'sec-preface-b4fc6291',
   'header_path': ['PREFACE'],
   'source_span': 'Bonham ISD Middle School Additions and Renovations Bonham, Texas',
   'confidence': 1.0}},
 {'id': '1',
  'entity': {'type': 'elevator',
   'name': 'hydraulic passenger elevators',
   'manufacturer': None},
  'attribute': {'raw': 'hydraulic passenger elevators', 'canonical': None},
  'value': {'raw': 'hydraulic passenger elevators',
   'type': 'string',
   'num': None,
   'unit': None,
   'min': None,
   'max': None},
  'op': '=',
  'qualifiers': None,
  'context': {'doc_i

##### Pass D — Normalize/Canonicalize (**optional**, might require testing and bechmarking)
 *	A tiny YAML attribute catalog (grow it over time)
 *	A canonicalizer that maps `attribute.raw → attribute.canonical` **using synonyms/regex** + a safety threshold
 *	Range/inequality parsing to convert `value.raw` into structured `{op, num/unit, min/max, tolerance}`

**Lightweight Attribute Catalog (grow it over time)**

> We don’t want a rigid schema, but we do want canonical handles for matching.

```yaml
# attributes.catalog.yaml
attributes:
  rated_load:
    synonyms: ["rated load", "capacity", "rated capacity", "load capacity"]
    value_type: quantity
    unit_hints: ["lb", "kg"]

  rated_speed_up:
    synonyms: ["rated speed up", "up speed", "speed (up)"]
    value_type: quantity
    unit_hints: ["fpm", "m/s"]

  rated_speed_down:
    synonyms: ["rated speed down", "down speed", "speed (down)"]
    value_type: quantity
    unit_hints: ["fpm", "m/s"]
...
```

In [ ]:
import re, json, yaml, math
from difflib import SequenceMatcher
from typing import List, Dict, Any, Optional, Tuple, Union
from dataclasses import dataclass

# ---------- Catalog structures

# @dataclass
# class AttrEntry:
#     canonical: str
#     synonyms: List[Union[str, re.Pattern]]
#     value_type: Optional[str] = None
#     unit_hints: Optional[List[str]] = None

# def _norm(s: str) -> str:
#     return re.sub(r'\s+', ' ', s.strip().lower())

# def load_attribute_catalog(path: str) -> List[AttrEntry]:
#     with open(path, "r", encoding="utf-8") as fp:
#         y = yaml.safe_load(fp) or {}
#     out: List[AttrEntry] = []
#     for canon, spec in (y.get("attributes") or {}).items():
#         syns = []
#         for item in spec.get("synonyms", []):
#             if isinstance(item, dict) and "regex" in item:
#                 syns.append(re.compile(item["regex"], re.IGNORECASE))
#             else:
#                 syns.append(str(item))
#         out.append(AttrEntry(
#             canonical=canon,
#             synonyms=syns,
#             value_type=spec.get("value_type"),
#             unit_hints=spec.get("unit_hints"),
#         ))
#     return out

# ---------- Attribute matching

# def _token_overlap(a: str, b: str) -> float:
#     A = set(re.findall(r"[a-z0-9]+", _norm(a)))
#     B = set(re.findall(r"[a-z0-9]+", _norm(b)))
#     if not A or not B: return 0.0
#     return len(A & B) / len(A | B)

# def _best_canonical_for(raw: str, header_path: List[str], catalog: List[AttrEntry]) -> Tuple[Optional[str], float, Optional[AttrEntry]]:
#     """
#     Returns (canonical, score, entry). Score in [0..1]. We use:
#     1) regex hit = 1.0
#     2) exact synonym match = 0.98
#     3) token-overlap + fuzzy ratio blend (>= 0.66 accepted)
#     Small boost if header_path contains a word from canonical key.
#     """
#     raw_norm = _norm(raw)
#     best = (None, 0.0, None)

#     for entry in catalog:
#         # regex synonyms
#         for syn in entry.synonyms:
#             if isinstance(syn, re.Pattern):
#                 if syn.search(raw):
#                     score = 1.0
#                     if score > best[1]: best = (entry.canonical, score, entry)
#                 continue

#             syn_norm = _norm(syn)
#             if raw_norm == syn_norm:
#                 score = 0.98
#             elif syn_norm in raw_norm or raw_norm in syn_norm:
#                 # substring containment
#                 score = 0.9
#             else:
#                 # blend token-overlap and fuzzy
#                 tok = _token_overlap(raw_norm, syn_norm)
#                 fuzz = SequenceMatcher(None, raw_norm, syn_norm).ratio()
#                 score = 0.5 * tok + 0.5 * fuzz  # simple blend

#             # header_path prior (mild)
#             hp = " ".join(header_path).lower()
#             if entry.canonical.split("_")[0] in hp:
#                 score += 0.05

#             if score > best[1]:
#                 best = (entry.canonical, score, entry)

#     # accept only if reasonably confident
#     if best[1] >= 0.66:
#         return best
#     return (None, best[1], None)

# def canonicalize_attributes(facts: List[Dict[str, Any]], catalog: List[AttrEntry]) -> List[Dict[str, Any]]:
#     out = []
#     for f in facts:
#         raw_attr = (f.get("attribute") or {}).get("raw") or ""
#         header_path = (f.get("context") or {}).get("header_path") or []
#         canon, score, entry = _best_canonical_for(raw_attr, header_path, catalog)
#         if canon:
#             f["attribute"]["canonical"] = canon
#             # optionally stamp expected value_type/unit_hints for downstream checks
#             if entry and entry.value_type and not f["value"].get("type"):
#                 f["value"]["type"] = entry.value_type
#             if entry and entry.unit_hints and not f["value"].get("unit"):
#                 # only hint; do not overwrite
#                 f["qualifiers"] = f.get("qualifiers") or {}
#                 f["qualifiers"].setdefault("unit_hints", entry.unit_hints)
            
#             # Ensure qualifiers is a dict and set canonical_score
#             f["qualifiers"] = f.get("qualifiers") or {}
#             f["qualifiers"]["canonical_score"] = round(score, 3)
#         out.append(f)
#     return out

# ---------- Range & inequality parsing

# Pre-compiled patterns (cover common spec language)
P_THROUGH = re.compile(r'\b(?:between|from)\s+([0-9]+(?:\.[0-9]+)?)\s*(\w+)?\s+(?:to|and|-|–|—)\s*([0-9]+(?:\.[0-9]+)?)\s*(\w+)?', re.I)
P_RANGE_DASH = re.compile(r'\b([0-9]+(?:\.[0-9]+)?)\s*(\w+)?\s*[–—-]\s*([0-9]+(?:\.[0-9]+)?)\s*(\w+)?')
P_GE = re.compile(r'\b(?:≥|>=|not less than|minimum|min\.|at least)\b', re.I)
P_LE = re.compile(r'\b(?:≤|<=|not more than|maximum|max\.|no more than|up to|not to exceed|nte)\b', re.I)
P_GT = re.compile(r'\b(?:>|greater than|more than)\b', re.I)
P_LT = re.compile(r'\b(?:<|less than)\b', re.I)
P_PLUSMINUS = re.compile(r'([0-9]+(?:\.[0-9]+)?)\s*(\w+)?\s*(?:±|\+/-)\s*([0-9]+(?:\.[0-9]+)?)\s*(\w+)?')

# Find a primary (num, unit) pair
P_NUMUNIT = re.compile(r'([0-9]+(?:\.[0-9]+)?)\s*([a-zA-Z%/]+)?')

# Alt value in parentheses, e.g. "42 inches (1067 mm)"
P_ALT_PARENS = re.compile(r'\(([^)]+)\)')

def _first_num_unit(s: str) -> Tuple[Optional[float], Optional[str]]:
    m = P_NUMUNIT.search(s)
    if not m: return (None, None)
    num = float(m.group(1))
    unit = m.group(2).lower() if m.group(2) else None
    return (num, unit)

def parse_value_constraints(value_raw: str) -> Dict[str, Any]:
    """
    Parse a single value.raw into structured constraints:
    returns fields that you can merge back into f["value"]/f["op"]/f["qualifiers"].
    Handles: ranges (between X and Y, X–Y), >=, <=, >, <, ± tolerance, alt units in parentheses.
    """
    s = value_raw.strip()

    # ± tolerance
    m = P_PLUSMINUS.search(s)
    if m:
        num = float(m.group(1)); unit = (m.group(2) or "").lower() or None
        tol = float(m.group(3)); tol_unit = (m.group(4) or "").lower() or unit
        return {
            "op": "~",
            "value": {"type": "quantity", "num": num, "unit": unit or tol_unit, "raw": value_raw},
            "qualifiers": {"tolerance": {"plus_minus": tol, "unit": tol_unit or unit}}
        }

    # between / from ... to ...
    m = P_THROUGH.search(s) or P_RANGE_DASH.search(s)
    if m:
        a = float(m.group(1)); a_u = (m.group(2) or "").lower() or None
        b = float(m.group(3)); b_u = (m.group(4) or "").lower() or None
        unit = a_u or b_u  # prefer first if present
        lo, hi = (a, b) if a <= b else (b, a)
        return {
            "op": "between",
            "value": {"type": "range", "min": lo, "max": hi, "unit": unit, "raw": value_raw}
        }

    # inequalities (>=, <=, >, <)
    if P_GE.search(s):
        num, unit = _first_num_unit(s)
        return {"op": ">=", "value": {"type": "quantity", "num": num, "unit": unit, "raw": value_raw}}
    if P_LE.search(s):
        num, unit = _first_num_unit(s)
        return {"op": "<=", "value": {"type": "quantity", "num": num, "unit": unit, "raw": value_raw}}
    if P_GT.search(s):
        num, unit = _first_num_unit(s)
        return {"op": ">", "value": {"type": "quantity", "num": num, "unit": unit, "raw": value_raw}}
    if P_LT.search(s):
        num, unit = _first_num_unit(s)
        return {"op": "<", "value": {"type": "quantity", "num": num, "unit": unit, "raw": value_raw}}

    # plain quantity (fallback)
    num, unit = _first_num_unit(s)
    if num is not None:
        out = {"op": "=", "value": {"type": "quantity", "num": num, "unit": unit, "raw": value_raw}}
    else:
        out = {"op": "=", "value": {"type": "text", "raw": value_raw}}

    # alt units in parentheses → stash in qualifiers.alt_values
    alts = []
    for m in P_ALT_PARENS.finditer(s):
        # naive parse "1067 mm" inside the parens
        n2, u2 = _first_num_unit(m.group(1))
        if n2 is not None:
            alts.append({"num": n2, "unit": u2})
    if alts:
        out.setdefault("qualifiers", {})
        out["qualifiers"]["alt_values"] = alts
    return out

def apply_ranges_inequalities(facts: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for f in facts:
        v = f.get("value", {})
        raw = v.get("raw") or ""
        parsed = parse_value_constraints(raw)
        # merge: keep original raw, but update structured fields
        f["op"] = parsed.get("op", f.get("op", "="))
        # keep both raw + structured
        f["value"]["type"] = parsed["value"]["type"]
        # copy numeric fields if present
        for k in ["num", "unit", "min", "max"]:
            if k in parsed["value"] and parsed["value"][k] is not None:
                f["value"][k] = parsed["value"][k]
        # add qualifiers if any
        if "qualifiers" in parsed:
            # Ensure qualifiers is a dict (handle None case)
            f["qualifiers"] = f.get("qualifiers") or {}
            f["qualifiers"].update(parsed["qualifiers"])
        out.append(f)
    return out

# ---------- Orchestrator

# def canonicalize_and_parse(
#     facts: List[Dict[str, Any]],
#     catalog_yaml_path: str
# ) -> List[Dict[str, Any]]:
#     catalog = load_attribute_catalog(catalog_yaml_path)
#     facts2 = canonicalize_attributes(facts, catalog)
#     facts3 = apply_ranges_inequalities(facts2)
#     return facts3

##### Trigger the canonicalization of the llm extracted facts (**optional**, requires more testing and bechmarking)

In [159]:
# Pass C: 
# First convert Fact objects back to dictionaries
# spec_facts = [f.model_dump() for f in spec_facts] if spec_facts and hasattr(spec_facts[0], 'model_dump') else spec_facts
# normalizes the facts: maps names, parses ranges/inequalities, pulls units/numbers.
#facts = canonicalize_and_parse(spec_facts, "./spec_attribute_catalog.yaml")

processed_spec_facts = apply_ranges_inequalities(spec_facts)  # no canonical mapping
# (Optional) Now normalize units with your pint step from earlier:
# spec_facts = normalize_units([Fact.model_validate(f) for f in spec_facts])  # Convert to Fact objects for normalization

# Convert back to dictionaries for JSON serialization
# spec_facts = [f.model_dump() for f in spec_facts]

# Write out:
# jsonl_file = PARSED_DATA_DIR / "spec.facts.canonical.jsonl"
# with open(jsonl_file,"w",encoding="utf-8") as fp:
#     for f in spec_facts:
#         fp.write(json.dumps(f, ensure_ascii=False) + "\n")

In [157]:
spec_facts[40]

{'id': '7',
 'entity': {'type': 'elevator', 'name': None, 'manufacturer': None},
 'attribute': {'raw': 'Hall Fixtures', 'canonical': 'hall fixtures'},
 'value': {'raw': 'Satin stainless steel, No. 4 finish',
  'type': 'quantity',
  'num': 4.0,
  'unit': 'finish',
  'min': None,
  'max': None},
 'op': '=',
 'qualifiers': None,
 'context': {'doc_id': 'Spec 14 24 00 - Hydraulic Elevators_redacted.pdf',
  'section_id': 'sec-part-2-products-2345-d3ae75db',
  'header_path': ['PART 2 - PRODUCTS', '2345'],
  'source_span': 'Hall Fixtures Satin stainless steel, No. 4 finish.',
  'confidence': 1.0}}

In [213]:
len(processed_spec_facts)

112

In [158]:
processed_spec_facts[40]

{'id': '7',
 'entity': {'type': 'elevator', 'name': None, 'manufacturer': None},
 'attribute': {'raw': 'Hall Fixtures', 'canonical': 'hall fixtures'},
 'value': {'raw': 'Satin stainless steel, No. 4 finish',
  'type': 'quantity',
  'num': 4.0,
  'unit': 'finish',
  'min': None,
  'max': None},
 'op': '=',
 'qualifiers': None,
 'context': {'doc_id': 'Spec 14 24 00 - Hydraulic Elevators_redacted.pdf',
  'section_id': 'sec-part-2-products-2345-d3ae75db',
  'header_path': ['PART 2 - PRODUCTS', '2345'],
  'source_span': 'Hall Fixtures Satin stainless steel, No. 4 finish.',
  'confidence': 1.0}}

## Submittal RAG

### PDF loading (with Docling)

In [72]:
# Data Loading
ocr_converter = DocumentConverter(format_options=create_docling_config(full_page_ocr=True))
submittal_file = Path(DATA_DIR / "Submittal and Product Description_redacted.pdf")
submittal_doc = ocr_converter.convert(source=submittal_file)

2025-10-18 17:28:57,436 - docling.datamodel.document - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-18 17:28:57,459 - docling.document_converter - INFO - Going to convert document batch...
2025-10-18 17:28:57,462 - docling.document_converter - INFO - Initializing pipeline for StandardPdfPipeline with options hash c6e870fd3b31b69322ce46d01ce74a48
/Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/.venv/lib/python3.13/site-packages/docling/models/easyocr_model.py:68: UserWarning: Deprecated field. Better to set the `accelerator_options.device` in `pipeline_options`. When `use_gpu and accelerator_options.device == AcceleratorDevice.CUDA` the GPU is used to run EasyOCR. Otherwise, EasyOCR runs in CPU.
  warnings.warn(
2025-10-18 17:29:03,085 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-18 17:29:06,158 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-18 17:29:06,826 - docling.pipeline.base_pipeline - 

### Chunking (unsing Docling HybridChunker)

In [74]:
import tiktoken
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer

# Chunking
tokenizer = OpenAITokenizer(
    tokenizer=tiktoken.encoding_for_model("gpt-4o"),
    max_tokens=128 * 1024,  # context window length required for OpenAI tokenizers
)

# table serializer that serializes tables to Markdown instead of the triplet notation used by default
class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(doc=doc, table_serializer=MarkdownTableSerializer())

chunker = HybridChunker(tokenizer=tokenizer, serializer_provider=MDTableSerializerProvider())
chunk_iter = chunker.chunk(dl_doc=submittal_doc.document)
chunks = list(chunk_iter)

logger.info(f"Submittal Document Chunks: {len(chunks)}")

2025-10-18 17:43:07,371 - __main__ - INFO - Submittal Document Chunks: 59


In [80]:
#chunks[0].text

# extract text and metadata from chunks
submittal_documents, submittal_metadatas = [], []
for chunk in chunks:
    submittal_documents.append(chunk.text)
    submittal_metadatas.append(chunk.meta.export_json_dict())

#### Vectorization (using Qdrant inMemory)

##### catalog-driven, domain-agnostic metadata tagger

In [89]:
import re, yaml
from typing import Dict, Any, List, Optional

# --- load your Pass C YAML catalog (attributes + synonyms/regex)
def load_attribute_catalog(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as fp:
        return yaml.safe_load(fp) or {}

# --- compile synonym patterns once
def compile_attr_patterns(catalog: Dict[str, Any]):
    pats = {}
    for canon, spec in (catalog.get("attributes") or {}).items():
        pats[canon] = []
        for syn in spec.get("synonyms", []):
            if isinstance(syn, dict) and "regex" in syn:
                pats[canon].append(re.compile(syn["regex"], re.I))
            else:
                # escape and allow whitespace variants
                patt = re.compile(r"\b" + re.sub(r"\s+", r"\\s+", re.escape(syn)) + r"\b", re.I)
                pats[canon].append(patt)
    return pats

# --- generic detectors (no product bias)
ORG_SUFFIX = r"(?:inc\.?|llc|l\.?l\.?c\.?|corp\.?|corporation|co\.|company|ltd\.?|gmbh|s\.?a\.?)"
RE_MANUF_LINE = re.compile(r"\b(manufacturer|by|supplier)\s*[:\-]\s*(.+)$", re.I | re.M)
RE_ORG = re.compile(r"\b([A-Z][A-Za-z0-9&\.\- ]+)\s+" + ORG_SUFFIX + r"\b", re.I)
RE_MODEL = re.compile(r"\b(model|series|type|product\s*code|catalog\s*no\.?|cat\.\s*no\.?|part\s*no\.?|sku)\s*[:\-#]\s*([A-Za-z0-9\-\._/]+)", re.I)
RE_STANDARDS = re.compile(r"\b(ISO|IEC|ASME|ASTM|ANSI|NFPA|UL|CSA|EN|NEMA|IEEE)\s*[-: ]?\s*([A-Z0-9\.\-]+)", re.I)
RE_CSI_SECTION = re.compile(r"\bSECTION\s+(\d{2}\s*\d{2}\s*\d{2})\b|\bDIVISION\s+(\d{2})\b", re.I)

# numbers + units (very broad)
UNIT = r"(?:mm|cm|m|in|inch|inches|ft|feet|yd|kg|lb|lbs|g|N|kN|Pa|kPa|MPa|psi|bar|C|F|V|VAC|VDC|A|amp[s]?|Hz|kW|W|hp|rpm|rps|fpm|m/s|gpm|l/s|L/s|cfm|scfm|°[CF])"
RE_NUMUNIT = re.compile(r"(-?\d{1,3}(?:[\d,]{0,3})?(?:\.\d+)?)\s*(" + UNIT + r")\b", re.I)

# section/title heuristics for tiering (generic)
TIER_MAP = [
    (re.compile(r"\b(shop\s*drawing|submittal|product\s*data|data\s*sheet|cut\s*sheet)\b", re.I), "submittal"),
    (re.compile(r"\b(specification[s]?|technical\s*data|specs|layout|power\s*data|wiring\s*diagram)\b", re.I), "technical"),
    (re.compile(r"\b(brochure|marketing|sales)\b", re.I), "brochure"),
]

def detect_source_tier(text: str) -> str:
    head = text[:400]
    for rx, tier in TIER_MAP:
        if rx.search(head):
            return tier
    return "unknown"

def dedupe_keep_order(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x and x not in seen:
            seen.add(x); out.append(x)
    return out

def tag_payload_generic(text: str, attr_patterns: Dict[str, List[re.Pattern]]) -> Dict[str, Any]:
    """
    Domain-agnostic metadata to store in Qdrant payload.
    - attributes_present: which canonical attributes (from your YAML) are likely discussed in this chunk
    - manufacturer_candidates: org-like names or explicit 'Manufacturer:'
    - model_tokens: model/series/type/product code values
    - standards: list of (org, code) pairs as 'ORG CODE'
    - numbers_units: compact tokens '120 fpm', '42 in', etc. (for filtering/debug)
    - csi: section/division hints if present
    - source_tier: rough tier for tie-breaking
    """
    md = {}

    # attributes present (catalog-driven)
    attrs = []
    for canon, pats in attr_patterns.items():
        if any(p.search(text) for p in pats):
            attrs.append(canon)
    md["attributes_present"] = attrs

    # manufacturer candidates
    mans = []
    for m in RE_MANUF_LINE.finditer(text):
        mans.append(m.group(2).strip())
    for m in RE_ORG.finditer(text):
        mans.append(m.group(0).strip())
    md["manufacturer_candidates"] = dedupe_keep_order(mans)

    # model / series / codes
    models = [m.group(2).strip() for m in RE_MODEL.finditer(text)]
    md["model_tokens"] = dedupe_keep_order(models)

    # standards
    stds = [f"{m.group(1).upper()} {m.group(2).upper()}" for m in RE_STANDARDS.finditer(text)]
    md["standards"] = dedupe_keep_order(stds)

    # numbers + units
    nums = [f"{m.group(1)} {m.group(2)}" for m in RE_NUMUNIT.finditer(text)]
    md["numbers_units"] = dedupe_keep_order(nums)
    md["unit_set"] = dedupe_keep_order([u.split()[-1].lower() for u in nums])

    # CSI hints
    sec = RE_CSI_SECTION.search(text)
    md["csi_section"] = (sec.group(1) or sec.group(2)) if sec else None

    # tier
    md["source_tier"] = detect_source_tier(text)

    return md

In [ ]:
from langchain_core.documents import Document
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

CSI_SPEC_ATTR_CATALOG_FILE_PATH = "spec_attributes_catalog.yaml"
# Embedding
embedding_dim = 1024 #max: 1536
# text-embedding-3-small can be used in a hybrid search system with BM25, 
# where both are used to improve search results by combining semantic vector search with keyword-based ranking
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=embedding_dim)

# Retrieval
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="submittal_product_description",
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
)
qdrant_vectorstore = QdrantVectorStore(
    client=client,
    collection_name="submittal_product_description",
    embedding=embedding_model,
)

# Enrich document chunks with custom metadata
# load catalog and compile patterns once
catalog = load_attribute_catalog(CSI_SPEC_ATTR_CATALOG_FILE_PATH)
attr_patterns = compile_attr_patterns(catalog)

def to_documents(chunks: List[str]) -> List[Document]:
    docs = []
    for i, txt in enumerate(chunks):
        payload = tag_payload_generic(txt, attr_patterns)
        docs.append(Document(page_content=txt, metadata=payload | {"chunk_id": i}))
    return docs

# build docs and add to Qdrant
docs = to_documents(submittal_documents)     # your list[str] chunks
_ = qdrant_vectorstore.add_documents(docs)    # instead of add_texts(...)

# add documents to qdrant
#_ = qdrant_vectorstore.add_texts(submittal_documents, submittal_metadatas)

# Start with a Naive retriever
qdrant_retriever = qdrant_vectorstore.as_retriever(search_kwargs={"k": 8})


2025-10-18 21:12:22,196 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-18 21:13:42,879 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


##### Add support for Sparse Vectors with BM25 ranking to our Qdrand Collection

In [ ]:
from qdrant_client.models import VectorParams, SparseVectorParams, Modifier, PointStruct, SparseVector
from fastembed import SparseTextEmbedding

# Initialize BM25 model
bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
from qdrant_client.models import VectorParams, SparseVectorParams, Modifier, PointStruct
from qdrant_client import QdrantClient
from fastembed import SparseTextEmbedding

# Initialize BM25 model
bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")

def recreate_collection_with_hybrid_support(qdrant_client: QdrantClient, old_collection_name: str, new_collection_name: str = None):
    """Recreate collection with both dense and sparse vector support"""
    
    if new_collection_name is None:
        new_collection_name = f"{old_collection_name}_hybrid"
    
    # Get all existing documents
    try:
        all_points = qdrant_client.scroll(
            collection_name=old_collection_name,
            limit=10000,
            with_payload=True,
            with_vectors=True
        )[0]
        print(f"Retrieved {len(all_points)} existing documents")
    except Exception as e:
        print(f"Failed to retrieve documents: {e}")
        return False
    
    # Create new collection with hybrid support
    try:
        qdrant_client.create_collection(
            collection_name=new_collection_name,
            vectors_config={
                "dense": VectorParams(size=1024, distance="Cosine"),  # Your existing OpenAI embeddings
            },
            sparse_vectors_config={
                "bm25": SparseVectorParams(modifier=Modifier.IDF),  # BM25 sparse vectors
            }
        )
        print(f"Created new collection {new_collection_name} with hybrid support")
    except Exception as e:
        print(f"Failed to create new collection: {e}")
        return False
    
    # Generate BM25 embeddings for existing documents
    try:
        documents = [point.payload.get('page_content', '') for point in all_points]
        print(f"Generating BM25 embeddings for {len(documents)} documents...")
        bm25_embeddings = list(bm25_model.embed(documents))
        print(f"Generated {len(bm25_embeddings)} BM25 embeddings")
    except Exception as e:
        print(f"Failed to generate BM25 embeddings: {e}")
        return False
    
    # Create new points with both vector types
    try:
        new_points = []
        for point, bm25_emb in zip(all_points, bm25_embeddings):
            # Handle existing vector format
            existing_vector = point.vector
            if isinstance(existing_vector, dict):
                dense_vector = existing_vector.get("dense", existing_vector)
            else:
                dense_vector = existing_vector
            
            new_points.append(
                PointStruct(
                    id=point.id,
                    vector={
                        "dense": dense_vector,  # Keep existing OpenAI embeddings
                        "bm25": bm25_emb.as_object()  # Add BM25 sparse vectors
                    },
                    payload=point.payload
                )
            )
        
        # Insert new points
        qdrant_client.upsert(
            collection_name=new_collection_name,
            points=new_points
        )
        
        print(f"Successfully migrated {len(new_points)} documents to hybrid collection")
        return new_collection_name
        
    except Exception as e:
        print(f"Failed to migrate documents: {e}")
        return False

# Usage
qdrant_client = qdrant_vectorstore.client
old_collection_name = qdrant_vectorstore.collection_name

# Recreate collection with hybrid support
new_collection_name = recreate_collection_with_hybrid_support(qdrant_client, old_collection_name)

if new_collection_name:
    print(f"Successfully created hybrid collection: {new_collection_name}")
    
    # Update your vectorstore to use the new collection
    # You'll need to recreate your qdrant_vectorstore with the new collection
    print("Next step: Update your qdrant_vectorstore to use the new collection")
else:
    print("Failed to create hybrid collection")

Retrieved 59 existing documents
Created new collection submittal_product_description_hybrid with hybrid support
Generating BM25 embeddings for 59 documents...
Generated 59 BM25 embeddings
Successfully migrated 59 documents to hybrid collection
Successfully created hybrid collection: submittal_product_description_hybrid
Next step: Update your qdrant_vectorstore to use the new collection


##### BM25 search test

In [212]:
from qdrant_client.models import SparseVector

def _bm25_search(collection_name: str, query_text: str, limit: int = 5) -> List[Any]:
    """Test BM25 search on the hybrid collection"""
    try:
        # Generate BM25 embedding for query
        query_embedding = next(bm25_model.query_embed(query_text))
        
        # Create SparseVector object
        sparse_vector = SparseVector(
            indices=query_embedding.indices.tolist(),
            values=query_embedding.values.tolist()
        )
        
        # Use query_points with correct SparseVector format and using parameter
        search_results = qdrant_client.query_points(
            collection_name=collection_name,
            query=sparse_vector,
            using="bm25",  # Specify which sparse vector to use
            query_filter=None,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )
        
        # print(f"BM25 search results for '{query_text}':")
        # for i, result in enumerate(search_results.points):
        #     print(f"{i+1}. Score: {result.score:.4f}")
        #     print(f"   Content: {result.payload.get('page_content', '')[:100]}...")
        #     print()
        
        return search_results.points
        
    except Exception as e:
        print(f"BM25 search failed: {e}")
        return []

# Test with your query
test_query = "Hydraulic Fluid | elevator | made from vegetable oil"
_bm25_search(new_collection_name, test_query)

[ScoredPoint(id='5319dce01ec942388d61d9e3bc8e5790', version=0, score=7.5497002601623535, payload={'page_content': 'Designed with sustainability and energy efficiency in mind, the Schindler 3300 XL is up to 60% more efficient than a hydraulic elevator: Power Factor\n*Applies to front-opening 4,500 Ibs service configuration with tall cars and entrances_', 'metadata': {'attributes_present': [], 'manufacturer_candidates': [], 'model_tokens': [], 'standards': ['EN ERGY', 'EN TRANCES'], 'numbers_units': [], 'unit_set': [], 'csi_section': None, 'source_tier': 'unknown', 'chunk_id': 13}}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='65ab89fd64624204911f9100ab64c333', version=0, score=4.445916652679443, payload={'page_content': "|                        | Cab Width            | Cab Depth             |                  | Door Width       | Hoistway Width        | Hoistway Depth        | Hoistway Depth          |                                                      |          

##### Hybrid Search (Dense and Sparce) Test

In [195]:
def hybrid_search_separate(collection_name: str, query_text: str, limit: int = 5):
    """Perform hybrid search by combining separate dense and sparse vector searches"""
    
    try:
        # Generate both embeddings
        dense_query_vector = embedding_model.embed_query(query_text)
        sparse_query_vector = next(bm25_model.query_embed(query_text))
        
        # Create SparseVector object
        sparse_vector = SparseVector(
            indices=sparse_query_vector.indices.tolist(),
            values=sparse_query_vector.values.tolist()
        )
        
        # Perform dense vector search
        dense_results = qdrant_client.query_points(
            collection_name=collection_name,
            query=dense_query_vector,
            using="dense",  # Use dense vectors
            query_filter=None,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )
        
        # Perform sparse vector (BM25) search
        sparse_results = qdrant_client.query_points(
            collection_name=collection_name,
            query=sparse_vector,
            using="bm25",  # Use BM25 sparse vectors
            query_filter=None,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )
        
        print(f"Dense vector search results for '{query_text}':")
        for i, result in enumerate(dense_results.points):
            print(f"{i+1}. Score: {result.score:.4f} (Dense)")
            print(f"   Content: {result.payload.get('page_content', '')[:100]}...")
            print()
        
        print(f"BM25 search results for '{query_text}':")
        for i, result in enumerate(sparse_results.points):
            print(f"{i+1}. Score: {result.score:.4f} (BM25)")
            print(f"   Content: {result.payload.get('page_content', '')[:100]}...")
            print()
        
        return dense_results.points, sparse_results.points
        
    except Exception as e:
        print(f"Hybrid search failed: {e}")
        return [], []

# Test separate searches
test_query = "Hydraulic Fluid | elevator | made from vegetable oil"
dense_results, sparse_results = hybrid_search_separate(new_collection_name, test_query)

2025-10-19 19:07:20,104 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Dense vector search results for 'Hydraulic Fluid | elevator | made from vegetable oil':
1. Score: 0.4622 (Dense)
   Content: Designed with sustainability and energy efficiency in mind, the Schindler 3300 XL is up to 60% more ...

2. Score: 0.4275 (Dense)
   Content: - Power Factor
- Compact; lightweight and durable design
- Gearless machine saves energy and avoids ...

3. Score: 0.4072 (Dense)
   Content: where provided shall be Iit by permanently installed
Emergency Phone and Data Line: Conduit shall be...

4. Score: 0.4067 (Dense)
   Content: - Shall be an enclosed, externally operable, motor circuit switch, shall be listed and lockable in t...

5. Score: 0.3853 (Dense)
   Content: Steel cables used in elevators are relatively inelastic, requiring a traction sheave diameter of at ...

BM25 search results for 'Hydraulic Fluid | elevator | made from vegetable oil':
1. Score: 7.5497 (BM25)
   Content: Designed with sustainability and energy efficiency in mind, the Schindler 3300 XL is u

##### Combined results with Scoring

In [196]:
def hybrid_search_combined(collection_name: str, query_text: str, limit: int = 5, alpha: float = 0.7):
    """Combine dense and sparse search results with weighted scoring"""
    
    try:
        # Generate both embeddings
        dense_query_vector = embedding_model.embed_query(query_text)
        sparse_query_vector = next(bm25_model.query_embed(query_text))
        
        # Create SparseVector object
        sparse_vector = SparseVector(
            indices=sparse_query_vector.indices.tolist(),
            values=sparse_query_vector.values.tolist()
        )
        
        # Perform both searches
        dense_results = qdrant_client.query_points(
            collection_name=collection_name,
            query=dense_query_vector,
            using="dense",
            query_filter=None,
            limit=limit * 2,  # Get more results to combine
            with_payload=True,
            with_vectors=False
        )
        
        sparse_results = qdrant_client.query_points(
            collection_name=collection_name,
            query=sparse_vector,
            using="bm25",
            query_filter=None,
            limit=limit * 2,
            with_payload=True,
            with_vectors=False
        )
        
        # Combine and score results
        combined_results = combine_search_results_scored(
            dense_results.points, 
            sparse_results.points, 
            alpha=alpha,
            max_results=limit
        )
        
        print(f"Combined hybrid search results for '{query_text}':")
        for i, result in enumerate(combined_results):
            print(f"{i+1}. Score: {result['combined_score']:.4f} (Dense: {result['dense_score']:.4f}, BM25: {result['sparse_score']:.4f})")
            print(f"   Content: {result['payload'].get('page_content', '')[:100]}...")
            print()
        
        return combined_results
        
    except Exception as e:
        print(f"Hybrid search failed: {e}")
        return []

def combine_search_results_scored(dense_results, sparse_results, alpha=0.7, max_results=5):
    """Combine dense and sparse search results with weighted scoring"""
    
    # Create document ID to scores mapping
    doc_scores = {}
    
    # Add dense vector scores
    for result in dense_results:
        doc_id = result.id
        doc_scores[doc_id] = {
            'dense_score': result.score,
            'sparse_score': 0.0,
            'payload': result.payload,
            'doc_id': doc_id
        }
    
    # Add sparse vector scores
    for result in sparse_results:
        doc_id = result.id
        if doc_id in doc_scores:
            doc_scores[doc_id]['sparse_score'] = result.score
        else:
            doc_scores[doc_id] = {
                'dense_score': 0.0,
                'sparse_score': result.score,
                'payload': result.payload,
                'doc_id': doc_id
            }
    
    # Calculate combined scores
    for doc_id, scores in doc_scores.items():
        combined_score = (alpha * scores['dense_score'] + (1 - alpha) * scores['sparse_score'])
        scores['combined_score'] = combined_score
    
    # Sort by combined score and return top results
    sorted_results = sorted(
        doc_scores.values(), 
        key=lambda x: x['combined_score'], 
        reverse=True
    )
    
    return sorted_results[:max_results]

# Test combined hybrid search
test_query = "Hydraulic Fluid | elevator | made from vegetable oil"
combined_results = hybrid_search_combined(new_collection_name, test_query, alpha=0.7)

2025-10-19 19:10:15,079 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Combined hybrid search results for 'Hydraulic Fluid | elevator | made from vegetable oil':
1. Score: 2.5884 (Dense: 0.4622, BM25: 7.5497)
   Content: Designed with sustainability and energy efficiency in mind, the Schindler 3300 XL is up to 60% more ...

2. Score: 1.5748 (Dense: 0.3443, BM25: 4.4459)
   Content: |                        | Cab Width            | Cab Depth             |                  | Door Wi...

3. Score: 0.9445 (Dense: 0.4072, BM25: 2.1982)
   Content: where provided shall be Iit by permanently installed
Emergency Phone and Data Line: Conduit shall be...

4. Score: 0.8545 (Dense: 0.3638, BM25: 1.9994)
   Content: configure your elevator work in multi-story buildings. Schindler Plan was developed to enable accura...

5. Score: 0.8471 (Dense: 0.4067, BM25: 1.8746)
   Content: - Shall be an enclosed, externally operable, motor circuit switch, shall be listed and lockable in t...



#### Test RAG Chain

##### Prompt templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

RAG_SYSTEM_COMPARATOR_PROMPT = """
You are a construction/architecture spec assistant. You understand CSI specs and submittals.
Decide if a contractor submittal chunk supports a spec fact. Output JSON only.

Rules:
- No inference beyond the chunk text.
- Provide submittal_evidence as a ≤25-word verbatim substring of the chunk.
- For numeric facts: compare units and numbers; respect operators (=, <=, >=, between, ~).
- If the chunk is irrelevant or missing the value, verdict="unclear".
- JSON keys: verdict, reason, submittal_evidence.
"""

RAG_HUMAN_COMPARATOR_PROMPT_TEMPLATE = """
Spec fact:
{spec_fact_json}

Submittal chunk (metadata: tier={source_tier}, tags={tags}):
<<<
{submittal_chunk}
>>>

Return only:
{{"verdict":"consistent|inconsistent|unclear","reason":"...", "submittal_evidence":"..."}}
"""

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
llm_comparator_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_COMPARATOR_PROMPT),
    ("human", RAG_HUMAN_COMPARATOR_PROMPT_TEMPLATE),
])
json_parser = JsonOutputParser()  # expects a single JSON object

##### Submittal Query helpers

In [180]:
import json, re
from typing import List, Dict, Any, Optional, Tuple
from langchain_core.documents import Document

# ---- guardrails: evidence substring + simple numeric sanity (optional) ----

UNIT_SYNONYMS = {
    "fpm": ["fpm","ft/min","feet per minute"],
    "in": ["in","inch","inches","\""],
    "mm": ["mm","millimeter","millimeters"],
    "lb": ["lb","lbs","pound","pounds"],
    # add more as needed
}

# Optional policy keyed by canonical attribute (use if available)
POLICY_BY_CANONICAL = {
    # "rated_speed": {"abs": 5.0},
    # "door_width":  {"abs": 0.25},
}

DEFAULT_TOLERANCES_BY_UNIT = {
    "fpm": {"abs": 5.0},
    "in":  {"abs": 0.25},
    "mm":  {"abs": 2.0},
    "lb":  {"pct": 0.0},
}

_NUMUNIT = re.compile(r'(-?\d+(?:[\d,]*)(?:\.\d+)?)\s*([a-zA-Z/%°]+)?')

def _parse_num_unit(s: str) -> Optional[Tuple[float, str]]:
    m = _NUMUNIT.search((s or "").replace(",", ""))
    if not m: return None
    try: return float(m.group(1)), (m.group(2) or "").lower()
    except: return None

def evidence_ok(evidence: str, chunk_text: str) -> bool:
    return bool(evidence) and (evidence in chunk_text)

def numeric_sanity(spec_fact: Dict[str, Any], evidence: str,
                   tol_abs: float = 0.0, tol_pct: float = 0.0) -> bool:
    v = spec_fact.get("value", {})
    if v.get("type") not in {"quantity","range"}:
        return True
    parsed = _parse_num_unit(evidence)
    if v.get("type") == "range":
        if not parsed: return False
        x, _ = parsed
        lo, hi = float(v["min"]), float(v["max"])
        return (lo - tol_abs) <= x <= (hi + tol_abs)
    if not parsed: return False
    x, _ = parsed
    tgt = v.get("num")
    if tgt is None: return True
    tgt = float(tgt)
    op = spec_fact.get("op","=")
    if op == "=":  return abs(x - tgt) <= max(tol_abs, tol_pct * max(abs(x), abs(tgt), 1e-9))
    if op == ">=": return x + tol_abs >= tgt
    if op == "<=": return x - tol_abs <= tgt
    if op == ">":  return x > tgt
    if op == "<":  return x < tgt
    if op == "~":  return abs(x - tgt) <= max(tol_abs, tol_pct * max(abs(x), abs(tgt)))
    return True

def score_candidate(verdict: str, chunk_text: str, spec_fact: Dict[str, Any],
                    tier: str, evidence: str) -> float:
    s = 0.0
    s += 2.0 if verdict == "consistent" else 0.8 if verdict == "inconsistent" else 0.2
    s += 0.4 if tier in {"technical","submittal"} else 0.2 if tier == "brochure" else 0.0
    unit = (spec_fact.get("value") or {}).get("unit","")
    if unit and unit.lower() in (chunk_text or "").lower():
        s += 0.2
    if evidence: s += 0.2
    return s

# ---- catalog-driven synonyms + query terms (agnostic) ----
def _canon_synonyms(catalog: Dict[str, Any], spec_fact: Dict[str, Any]) -> List[str]:
    canon = spec_fact.get("attribute", {}).get("canonical")
    if not canon:
        return [spec_fact.get("attribute", {}).get("raw","")]
    syns = (catalog.get("attributes", {}).get(canon, {}) or {}).get("synonyms", [])
    out = []
    for s in syns:
        out.append(s["regex"] if isinstance(s, dict) and "regex" in s else str(s))
    return [s for s in out if s]

def build_query_terms(spec_fact: Dict[str, Any], synonyms: List[str]) -> str:
    attr = spec_fact["attribute"].get("canonical") or spec_fact["attribute"].get("raw","")
    val_raw = (spec_fact.get("value") or {}).get("raw","")
    ent = spec_fact.get("entity", {})
    tokens: List[str] = [attr] + [s for s in synonyms if s]
    if val_raw: tokens.append(val_raw)
    if spec_fact.get("value", {}).get("num") is not None and spec_fact["value"].get("unit"):
        tokens.append(f"{spec_fact['value']['num']} {spec_fact['value']['unit']}")
    for k in ("manufacturer","name","type"):
        if ent.get(k): tokens.append(str(ent[k]))
    return " | ".join(t for t in tokens if t)

def _derive_variants(raw_attr: str) -> List[str]:
    s = (raw_attr or "").strip()
    if not s: return []
    base = s.lower()
    # normalize separators / parentheses / colon labels
    base = re.sub(r"[\(\):]", " ", base)
    base = re.sub(r"\s+", " ", base).strip()
    tokens = {base}
    # add simple reorders around “rated”, “min”, “max”
    tokens.add(base.replace("rated ", ""))
    tokens.add(base.replace("minimum ", "").replace("max ", ""))
    # add hyphen/space variants
    tokens.add(base.replace("-", " "))
    tokens.add(base.replace(" ", ""))
    return [t for t in tokens if t]

def build_query_terms_open(spec_fact: Dict[str, Any],
                           catalog: Dict[str, Any] | None = None) -> str:
    attr_raw = spec_fact["attribute"].get("raw","")
    terms = _derive_variants(attr_raw)

    # units / numbers
    v = spec_fact.get("value", {})
    if v.get("num") is not None and v.get("unit"):
        terms.append(f"{v['num']} {v['unit']}")
        terms.extend(UNIT_SYNONYMS.get(v["unit"].lower(), []))
    elif v.get("raw"):
        terms.append(v["raw"])

    # entity hints if present
    ent = spec_fact.get("entity", {})
    for k in ("manufacturer","name","type"):
        if ent.get(k): terms.append(str(ent[k]))

    # include catalog synonyms if available but don’t require canonical
    if catalog:
        canon = spec_fact["attribute"].get("canonical")
        if canon and canon in (catalog.get("attributes") or {}):
            syns = catalog["attributes"][canon].get("synonyms", [])
            for s in syns:
                terms.append(s["regex"] if isinstance(s, dict) and "regex" in s else str(s))

    return " | ".join(sorted({t for t in terms if t}))


def tolerances_for_fact(spec_fact: dict,
                        policy_by_canonical: dict | None = None,
                        default_by_unit: dict | None = None) -> tuple[float, float]:
    policy_by_canonical = policy_by_canonical or {}
    default_by_unit = default_by_unit or DEFAULT_TOLERANCES_BY_UNIT

    # 1) explicit tolerance in the fact (from “±” parse) wins
    tol = (spec_fact.get("qualifiers") or {}).get("tolerance")
    if tol and "plus_minus" in tol:
        return float(tol["plus_minus"]), 0.0

    # 2) policy by canonical
    canon = spec_fact.get("attribute", {}).get("canonical")
    if canon and canon in policy_by_canonical:
        t = policy_by_canonical[canon]
        return float(t.get("abs", 0.0)), float(t.get("pct", 0.0))

    # 3) fallback by unit
    unit = (spec_fact.get("value") or {}).get("unit", "")
    t = default_by_unit.get((unit or "").lower(), {})
    return float(t.get("abs", 0.0)), float(t.get("pct", 0.0))

##### RAG Chain

In [210]:
from typing_extensions import TypedDict
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document

# State definition
class State(TypedDict, total=False):
    spec_fact: Dict[str, Any]
    catalog: Dict[str, Any]
    top_k: int
    # tol_abs: float
    # tol_pct: float
    query: str
    candidates: List[Document]
    result: Dict[str, Any]

def _similarity_search_with_score(vs, query: str, k: int) -> List[Tuple[Document, float]]:
    try:
        return vs.similarity_search_with_score(query=query, k=k)
    except TypeError:
        # fallback (scores 0.0)
        docs = vs.similarity_search(query=query, k=k)
        return [(d, 0.0) for d in docs]

def naive_retrieval(state: State) -> State:
    spec_fact = state["spec_fact"]
    catalog = state["catalog"]
    top_k = state.get("top_k", 3)

    # query = build_query_terms(spec_fact, _canon_synonyms(catalog, spec_fact))
    query = build_query_terms_open(spec_fact, catalog)
    docs_scores = _similarity_search_with_score(qdrant_vectorstore, query, k=top_k)
    candidates = [d for d, _ in docs_scores]

    return {**state, "query": query, "candidates": candidates}

def bm25_retrieval(state: State) -> State:
    spec_fact = state["spec_fact"]
    catalog = state["catalog"]
    top_k = state.get("top_k", 3)

    # query = build_query_terms(spec_fact, _canon_synonyms(catalog, spec_fact))
    query = build_query_terms_open(spec_fact, catalog)
    results = _bm25_search(new_collection_name, query, limit=top_k)
    candidates = [Document(page_content=d.payload.get('page_content', ''), metadata=d.payload.get('metadata', {})) for d in results]

    return {**state, "query": query, "candidates": candidates}

def _compare_one(spec_fact: Dict[str, Any], doc: Document) -> Dict[str, Any]:
    md = doc.metadata or {}
    inputs = {
        "spec_fact_json": json.dumps(spec_fact, ensure_ascii=False),
        "source_tier": md.get("source_tier", "unknown"),
        "tags": ", ".join(md.get("attributes_present", []) or md.get("unit_set", []) or md.get("numbers_units", []) or []),
        "submittal_chunk": doc.page_content
    }
    raw = (llm_comparator_prompt | llm).invoke(inputs).content
    try:
        out = json.loads(raw)
    except Exception:
        out = json_parser.parse(raw)
    return out

def generate(state: State) -> State:
    spec_fact = state["spec_fact"]
    # per-fact tolerances
    tol_abs, tol_pct = tolerances_for_fact(
        spec_fact,
        policy_by_canonical=POLICY_BY_CANONICAL,
        default_by_unit=DEFAULT_TOLERANCES_BY_UNIT
    )

    ranked: List[Tuple[float, Dict[str, Any], Document]] = []
    for doc in state.get("candidates", []):
        out = _compare_one(spec_fact, doc)
        evidence = out.get("submittal_evidence", "") or ""
        # Guardrail 1: evidence must be substring
        if not evidence_ok(evidence, doc.page_content):
            continue
        # Guardrail 2: numeric sanity (optional but recommended)
        if (spec_fact.get("value") or {}).get("type") in {"quantity","range"}:
            if not numeric_sanity(spec_fact, evidence, tol_abs=tol_abs, tol_pct=tol_pct):
                # allow "inconsistent" results to remain; just score with low weight
                pass
        s = score_candidate(out.get("verdict","unclear"), doc.page_content, spec_fact,
                            doc.metadata.get("source_tier","unknown"), evidence)
        ranked.append((s, out, doc))

    if not ranked:
        return {**state, "result": {"verdict":"gap","reason":"no supporting evidence found","submittal_evidence":""}}

    ranked.sort(key=lambda x: x[0], reverse=True)
    best_score, best_out, best_doc = ranked[0]
    best = best_out | {
        "chunk_meta": best_doc.metadata,
        "chunk_preview": (best_doc.page_content[:280] + ("..." if len(best_doc.page_content) > 280 else "")),
        "query_used": state["query"]
    }
    return {**state, "result": best}

# Compose the chain
rag_naive_retriveval_chain = RunnableLambda(naive_retrieval) | RunnableLambda(generate)
rag_bm25_retriveval_chain = RunnableLambda(bm25_retrieval) | RunnableLambda(generate)

#### Build retrieval terms from spec facts (catalog-aware)

In [162]:
spec_json_fact = spec_facts[47]
spec_json_fact

{'id': '5',
 'entity': {'type': 'elevator',
  'name': 'Hydraulic Fluid',
  'manufacturer': None},
 'attribute': {'raw': 'made from vegetable oil', 'canonical': 'base material'},
 'value': {'raw': 'vegetable oil',
  'type': 'text',
  'num': None,
  'unit': None,
  'min': None,
  'max': None},
 'op': '=',
 'qualifiers': None,
 'context': {'doc_id': 'Spec 14 24 00 - Hydraulic Elevators_redacted.pdf',
  'section_id': 'sec-part-2-products-2-4-systems-and-components-21793e74',
  'header_path': ['PART 2 - PRODUCTS', '2.4 SYSTEMS AND COMPONENTS'],
  'source_span': 'made from vegetable oil',
  'confidence': 0.6}}

#### Comparator

In [217]:
# Minimal example spec_fact (use your Pass B/C output)

# Load your attribute catalog dict
CSI_SPEC_ATTR_CATALOG_FILE_PATH="./spec_attributes_catalog.yaml"
catalog = yaml.safe_load(open(CSI_SPEC_ATTR_CATALOG_FILE_PATH))

print(f"# Spec Facts: {len(processed_spec_facts)}")
comparator_results = []
# Run the chain
for spec_json_fact in processed_spec_facts:
    state_in = {
        "spec_fact": spec_json_fact,
        "catalog": catalog,
        "top_k": 4,
    }
    # result_state = rag_naive_retriveval_chain.invoke(state_in)
    result_state = rag_bm25_retriveval_chain.invoke(state_in)
    comparator_results.append(result_state["result"])
    print(json.dumps(result_state["result"], indent=2))

# Dump comparator_results to a file in JSON format for further inspection/debugging.
# Ensure the .vibe-log/ directory exists (as per general-rule)

comp_results_path = DATA_DIR / 'comparator_results.json'
try:
    with open(comp_results_path, 'w', encoding='utf-8') as f:
        json.dump(comparator_results, f, ensure_ascii=False, indent=2)
    logging.info(f"Comparator results dumped to {comp_results_path}")
except Exception as e:
    logging.error(f"Failed to write comparator_results to file: {str(e)}")



# Spec Facts: 112


2025-10-19 21:07:45,722 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:07:47,082 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:07:47,816 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:07:48,407 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{
  "verdict": "unclear",
  "reason": "No mention of location or address in the chunk.",
  "submittal_evidence": "no control closet is required. A 3-phase and 11Ov disconnect must be located in both the hoistway overhead and a location in the building outside of the hoistway",
  "chunk_meta": {
    "attributes_present": [],
    "manufacturer_candidates": [],
    "model_tokens": [],
    "standards": [],
    "numbers_units": [],
    "unit_set": [],
    "csi_section": null,
    "source_tier": "brochure",
    "chunk_id": 51
  },
  "chunk_preview": "- more information_\n- Up to four car group operation is available. Please consult with your local Schindler Sales Representative for more information:\n- Where permitted by code, no control closet is required. A 3-phase and 11Ov disconnect must be located in both the hoistway over...",
  "query_used": "Bonham ISD Middle School Additions and Renovations | Bonham, Texas | elevator | location"
}


2025-10-19 21:07:49,274 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:07:50,068 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:07:50,618 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:07:51,171 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{
  "verdict": "unclear",
  "reason": "No explicit mention of hydraulic passenger elevators in the chunk.",
  "submittal_evidence": "Designed with sustainability and energy efficiency in mind, the Schindler 3300 XL is up to 60% more efficient than a hydraulic elevator",
  "chunk_meta": {
    "attributes_present": [],
    "manufacturer_candidates": [],
    "model_tokens": [],
    "standards": [
      "EN ERGY",
      "EN TRANCES"
    ],
    "numbers_units": [],
    "unit_set": [],
    "csi_section": null,
    "source_tier": "unknown",
    "chunk_id": 13
  },
  "chunk_preview": "Designed with sustainability and energy efficiency in mind, the Schindler 3300 XL is up to 60% more efficient than a hydraulic elevator: Power Factor\n*Applies to front-opening 4,500 Ibs service configuration with tall cars and entrances_",
  "query_used": "elevator | hydraulic passenger elevators | hydraulicpassengerelevators"
}


...

{
  "verdict": "unclear",
  "reason": "Chunk lacks specific date or timing information related to Substantial Completion or warranty period.",
  "submittal_evidence": "- Shall be supplied and located in a building utility space outside the hoistway due to the Motor Controller being located in the Elevator Hoistway.",
  "chunk_meta": {
    "attributes_present": [
      "voltage",
      "horsepower"
    ],
    "manufacturer_candidates": [],
    "model_tokens": [],
    "standards": [
      "EN CLOSED",
      "NFPA 70.",
      "ASME A17.1.",
      "EN D",
      "NFPA 70"
    ],
    "numbers_units": [
      "60 Hz",
      "5000 Amps",
      "29.3 Amps",
      "26.2 Amps",
      "4.6 Amps",
      "-0.8 Amps",
      "70 Amps"
    ],
    "unit_set": [
      "hz",
      "amps"
    ],
    "csi_section": null,
    "source_tier": "unknown",
    "chunk_id": 1
  },
  "chunk_preview": "- Shall be an enclosed, externally operable, motor circuit switch, shall be listed and lockable in the open position

2025-10-19 21:13:48,386 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:48,945 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:49,508 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:49,930 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{
  "verdict": "unclear",
  "reason": "Chunk lacks specific maintenance duration details; no explicit mention of 12 months or full maintenance service.",
  "submittal_evidence": "12'-11 1 / 4 ' (3942)",
  "chunk_meta": {
    "attributes_present": [
      "rated_load"
    ],
    "manufacturer_candidates": [],
    "model_tokens": [],
    "standards": [],
    "numbers_units": [
      "9 ft"
    ],
    "unit_set": [
      "ft"
    ],
    "csi_section": null,
    "source_tier": "unknown",
    "chunk_id": 50
  },
  "chunk_preview": "|                        | Cab Width            | Cab Depth             |                  | Door Width       | Hoistway Width        | Hoistway Depth        | Hoistway Depth          |                                                      |                                        ...",
  "query_used": "12.0 months | elevator | initial maintenance service | initialmaintenanceservice"
}


2025-10-19 21:13:51,246 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:51,955 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:52,637 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:53,703 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-19 21:13:53,712 - root - INFO - Comparator results dumped to /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/comparator_results.json


{
  "verdict": "unclear",
  "reason": "No mention of emergency callback service in the chunk.",
  "submittal_evidence": "- IN US JURISDICTIONS ONKY,WHEN THE BUILDING SHALL PROVIDE SHUNT TRIP ACTIVATION OF DISCONNECTION OF ELECTRICAL POWER TO BOTH MAIN AND AUXILIARY ROWER CIRCUITS PRIOR TO SPRINKLER ACTIVATION (ASME A17.1-2007/CSA B44-07 RULE 2.8.3.3. ANDIQR LOCAL CODE):",
  "chunk_meta": {
    "attributes_present": [
      "rated_load",
      "voltage",
      "horsepower"
    ],
    "manufacturer_candidates": [
      "PRIOR TO SCHINDLER ELEVATOR COMPANY"
    ],
    "model_tokens": [
      "REAR"
    ],
    "standards": [
      "ASME A17.1-2007",
      "CSA B44-07",
      "EN CLOSURE",
      "CSA 38",
      "NFPA 72",
      "NFPA 13",
      "EN TRANCES",
      "UL 10B1.5",
      "ASME A17.1",
      "EN TRANCE"
    ],
    "numbers_units": [
      "18 M",
      "60 FT",
      "11 M",
      "36 FT",
      "1590 KG",
      "100 fpm",
      "2591 mm",
      "23.2 M",
      "1067 mm",
      "